<a href="https://colab.research.google.com/github/yeonsub/Learning_Material_for_Creation/blob/main/Clip_VQVAE_Flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F

import math
import itertools
from itertools import chain
import pandas as pd

from tqdm import tqdm
import albumentations as A

import timm
from transformers import DistilBertModel, DistilBertConfig, DistilBertTokenizer

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
from google.colab import userdata

# Colab 비밀에서 Kaggle 사용자 이름과 키를 불러옵니다.
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

### For Flickr 8k
!kaggle datasets download -d adityajn105/flickr8k
!unzip flickr8k.zip > /dev/null 2>&1
dataset = "8k"


### For Flickr 30k
# !kaggle datasets download -d hsankesara/flickr-image-dataset
# !unzip flickr-image-dataset.zip > /dev/null 2>&1
# dataset = "30k"


In [ ]:
if dataset == "8k":
  df = pd.read_csv("captions.txt")
  df['id'] = [id_ for id_ in range(df.shape[0] // 5) for _ in range(5)]
  df.to_csv("captions.csv", index=False)
  df = pd.read_csv("captions.csv")
  image_path = "/content/Images"
  captions_path = "/content"
elif dataset == "30k":
  df = pd.read_csv("/content/flickr30k_images/results.csv", delimiter="|")
  df.columns = ['image', 'caption_number', 'caption']
  df['caption'] = df['caption'].str.lstrip()
  df['caption_number'] = df['caption_number'].str.lstrip()
  df.loc[19999, 'caption_number'] = "4"
  df.loc[19999, 'caption'] = "A dog runs across the grass ."
  ids = [id_ for id_ in range(len(df) // 5) for _ in range(5)]
  df['id'] = ids
  df.to_csv("captions.csv", index=False)
  image_path = "/content/flickr30k_images/flickr30k_images"
  captions_path = "/content"

df.head()

In [ ]:
class CFG:
    debug = False
    image_path = image_path
    captions_path = captions_path
    batch_size = 32
    num_workers = 2
    head_lr = 1e-3
    image_encoder_lr = 1e-4
    text_encoder_lr = 1e-5
    weight_decay = 1e-3
    patience = 1
    factor = 0.8
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_name = 'resnet50'
    image_embedding = 2048
    text_encoder_model = "distilbert-base-uncased"
    text_embedding = 768
    text_tokenizer = "distilbert-base-uncased"
    max_length = 200

    pretrained = True # for both image encoder and text encoder
    trainable = True # for both image encoder and text encoder
    temperature = 1.0

    # image size
    size = 224

    # for projection head; used for both image and text encoders
    num_projection_layers = 1
    projection_dim = 256
    dropout = 0.1

    # Global tokenizer (will be initialized later)
    tokenizer = None

# Initialize CFG.tokenizer globally after the class definition
CFG.tokenizer = DistilBertTokenizer.from_pretrained(CFG.text_tokenizer)

In [ ]:
class AvgMeter:
    def __init__(self, name="Metric"):
        self.name = name
        self.reset()

    def reset(self):
        self.avg, self.sum, self.count = [0] * 3

    def update(self, val, count=1):
        self.count += count
        self.sum += val * count
        self.avg = self.sum / self.count

    def __repr__(self):
        text = f"{self.name}: {self.avg:.4f}"
        return text

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group["lr"]


In [ ]:
class CLIPDataset(torch.utils.data.Dataset):
    def __init__(self, image_filenames, captions, tokenizer, transforms):
        """
        image_filenames and cpations must have the same length; so, if there are
        multiple captions for each image, the image_filenames must have repetitive
        file names
        """

        self.image_filenames = image_filenames
        self.captions = list(captions)
        self.encoded_captions = tokenizer(
            list(captions), padding=True, truncation=True, max_length=CFG.max_length
        )
        self.transforms = transforms

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(values[idx])
            for key, values in self.encoded_captions.items()
        }

        image = cv2.imread(f"{CFG.image_path}/{self.image_filenames[idx]}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transforms(image=image)['image']
        item['image'] = torch.tensor(image).permute(2, 0, 1).float()
        item['caption'] = self.captions[idx]

        return item


    def __len__(self):
        return len(self.captions)


def get_transforms(mode="train"):
    if mode == "train":
        return A.Compose(
            [
                A.Resize(CFG.size, CFG.size),
                A.Normalize(max_pixel_value=255.0),
            ]
        )
    else:
        return A.Compose(
            [
                A.Resize(CFG.size, CFG.size),
                A.Normalize(max_pixel_value=255.0),
            ]
        )

In [ ]:
class ImageEncoder(nn.Module):
    """
    Encode images to a fixed size vector
    """

    def __init__(
        self, model_name=CFG.model_name, pretrained=CFG.pretrained, trainable=CFG.trainable
    ):
        super().__init__()
        self.model = timm.create_model(
            model_name, pretrained, num_classes=0, global_pool="avg"
        )
        for p in self.model.parameters():
            p.requires_grad = trainable

    def forward(self, x):
        return self.model(x)

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, model_name=CFG.text_encoder_model, pretrained=CFG.pretrained, trainable=CFG.trainable):
        super().__init__()
        if pretrained:
            self.model = DistilBertModel.from_pretrained(model_name)
        else:
            self.model = DistilBertModel(config=DistilBertConfig())

        for p in self.model.parameters():
            p.requires_grad = trainable

        # we are using the CLS token hidden representation as the sentence's embedding
        self.target_token_idx = 0

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = output.last_hidden_state
        return last_hidden_state[:, self.target_token_idx, :]

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(
        self,
        embedding_dim,
        projection_dim=CFG.projection_dim,
        dropout=CFG.dropout
    ):
        super().__init__()
        self.projection = nn.Linear(embedding_dim, projection_dim)
        self.gelu = nn.GELU()
        self.fc = nn.Linear(projection_dim, projection_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(projection_dim)

    def forward(self, x):
        projected = self.projection(x)
        x = self.gelu(projected)
        x = self.fc(x)
        x = self.dropout(x)
        x = x + projected
        x = self.layer_norm(x)
        return x

In [ ]:
class CLIPModel(nn.Module):
    def __init__(
        self,
        temperature=CFG.temperature,
        image_embedding=CFG.image_embedding,
        text_embedding=CFG.text_embedding,
    ):
        super().__init__()
        self.image_encoder = ImageEncoder()
        self.text_encoder = TextEncoder()
        self.image_projection = ProjectionHead(embedding_dim=image_embedding)
        self.text_projection = ProjectionHead(embedding_dim=text_embedding)
        self.temperature = temperature

    def forward(self, batch):
        # Getting Image and Text Features
        image_features = self.image_encoder(batch["image"])
        text_features = self.text_encoder(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
        )
        # Getting Image and Text Embeddings (with same dimension)
        image_embeddings = self.image_projection(image_features)
        text_embeddings = self.text_projection(text_features)

        # Calculating the Loss
        logits = (text_embeddings @ image_embeddings.T) / self.temperature
        images_similarity = image_embeddings @ image_embeddings.T
        texts_similarity = text_embeddings @ text_embeddings.T
        targets = F.softmax(
            (images_similarity + texts_similarity) / 2 * self.temperature, dim=-1
        )
        texts_loss = cross_entropy(logits, targets, reduction='none')
        images_loss = cross_entropy(logits.T, targets.T, reduction='none')
        loss =  (images_loss + texts_loss) / 2.0 # shape: (batch_size)
        return loss.mean()


def cross_entropy(preds, targets, reduction='none'):
    log_softmax = nn.LogSoftmax(dim=-1)
    loss = (-targets * log_softmax(preds)).sum(1)
    if reduction == "none":
        return loss
    elif reduction == "mean":
        return loss.mean()


In [ ]:
def make_train_valid_dfs():
    dataframe = pd.read_csv(f"{CFG.captions_path}/captions.csv")
    max_id = dataframe["id"].max() + 1 if not CFG.debug else 100
    image_ids = np.arange(0, max_id)
    np.random.seed(42)
    valid_ids = np.random.choice(
        image_ids, size=int(0.2 * len(image_ids)), replace=False
    )
    train_ids = [id_ for id_ in image_ids if id_ not in valid_ids]
    train_dataframe = dataframe[dataframe["id"].isin(train_ids)].reset_index(drop=True)
    valid_dataframe = dataframe[dataframe["id"].isin(valid_ids)].reset_index(drop=True)
    return train_dataframe, valid_dataframe


def build_loaders(dataframe, mode):
    transforms = get_transforms(mode=mode)
    dataset = CLIPDataset(
        dataframe["image"].values,
        dataframe["caption"].values,
        tokenizer=CFG.tokenizer, # Use CFG.tokenizer
        transforms=transforms,
    )
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
        shuffle=True if mode == "train" else False,
    )
    return dataloader

In [ ]:
clip_model = CLIPModel().to(CFG.device)
train_df, valid_df = make_train_valid_dfs()
train_loader = build_loaders(train_df, mode="train")
valid_loader = build_loaders(valid_df, mode="valid")

### 학습된 CLIP 모델이 있으면 불러오기

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/clip_models/best.pt" .

clip_model.load_state_dict(torch.load('best.pt', map_location=CFG.device))
clip_model.eval() # 모델을 평가 모드로 설정

print("CLIP 모델이 성공적으로 로드되었습니다.")

#CLIP 추가 학습하기

In [ ]:
def train_epoch(model, train_loader, optimizer, lr_scheduler, step):
    loss_meter = AvgMeter()
    tqdm_object = tqdm(train_loader, total=len(train_loader))
    for batch in tqdm_object:
        batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}
        loss = model(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == "batch":
            lr_scheduler.step()

        count = batch["image"].size(0)
        loss_meter.update(loss.item(), count)

        tqdm_object.set_postfix(train_loss=loss_meter.avg, lr=get_lr(optimizer))
    return loss_meter


def valid_epoch(model, valid_loader):
    loss_meter = AvgMeter()
    tqdm_object = tqdm(valid_loader, total=len(valid_loader))
    for batch in tqdm_object:
        batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}
        loss = model(batch)

        count = batch["image"].size(0)
        loss_meter.update(loss.item(), count)

        tqdm_object.set_postfix(valid_loss=loss_meter.avg)
    return loss_meter


def clip_train(epochs):
    params = [
        {"params": clip_model.image_encoder.parameters(), "lr": CFG.image_encoder_lr},
        {"params": clip_model.text_encoder.parameters(), "lr": CFG.text_encoder_lr},
        {"params": itertools.chain(
            clip_model.image_projection.parameters(), clip_model.text_projection.parameters()
        ), "lr": CFG.head_lr, "weight_decay": CFG.weight_decay}
    ]
    optimizer = torch.optim.AdamW(params, weight_decay=0.)
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=CFG.patience, factor=CFG.factor
    )
    step = "epoch"

    best_loss = float('inf')
    for epoch in range(epochs):
        print(f"Epoch: {epoch + 1}")
        clip_model.train()
        train_loss = train_epoch(clip_model, train_loader, optimizer, lr_scheduler, step)
        clip_model.eval()
        with torch.no_grad():
            valid_loss = valid_epoch(clip_model, valid_loader)

        if valid_loss.avg < best_loss:
            best_loss = valid_loss.avg
            torch.save(clip_model.state_dict(), "best.pt")
            print("Saved Best Model!")

        lr_scheduler.step(valid_loss.avg)

In [ ]:
clip_epochs=5
clip_train(clip_epochs)

### 학습된 CLIP 모델을 Google Drive에 저장

In [ ]:
os.makedirs('/content/drive/MyDrive/Colab Notebooks/clip_models/', exist_ok=True)
!cp best.pt "/content/drive/MyDrive/Colab Notebooks/clip_models/"

#CLIP 모델 Test

In [ ]:
def get_image_embeddings(valid_df):
    valid_loader = build_loaders(
        valid_df,
        mode="valid"
    )

    global clip_model
    clip_model.eval()
    valid_image_embeddings = []
    with torch.no_grad():
        for batch in tqdm(valid_loader):
            image_features = clip_model.image_encoder(batch["image"].to(CFG.device))
            image_embeddings = clip_model.image_projection(image_features)
            valid_image_embeddings.append(image_embeddings)
    return torch.cat(valid_image_embeddings)

image_embeddings = get_image_embeddings(valid_df)

In [ ]:
def find_matches(model, image_embeddings, query, image_filenames, n=9):
    encoded_query = CFG.tokenizer([query])
    batch = {
        key: torch.tensor(values).to(CFG.device)
        for key, values in encoded_query.items()
    }
    with torch.no_grad():
        text_features = model.text_encoder(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
        )
        text_embeddings = model.text_projection(text_features)

    image_embeddings_n = F.normalize(image_embeddings, p=2, dim=-1)
    text_embeddings_n = F.normalize(text_embeddings, p=2, dim=-1)
    dot_similarity = text_embeddings_n @ image_embeddings_n.T

    values, indices = torch.topk(dot_similarity.squeeze(0), n * 5)
    matches = [image_filenames[idx] for idx in indices[::5]]

    _, axes = plt.subplots(3, 3, figsize=(10, 10))
    for match, ax in zip(matches, axes.flatten()):
        image = cv2.imread(f"{CFG.image_path}/{match}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)
        ax.axis("off")

    plt.show()

In [ ]:
query = input("검색할 이미지에 대한 텍스트 설명을 입력하세요: ")
find_matches(clip_model,
             image_embeddings,
             query=query,
             image_filenames=valid_df['image'].values,
             n=9)

#VQVAE 모델


Start by setting up variables

In [ ]:
features = [3, 32, 64, 128] # Input channels 3 for RGB images, increased feature depth
latent_channels = 16 # Increased latent channels to allow more information capacity
batch_size    = 128
beta_vae=0.05 # Adjusted beta_vae from trial error

# Assuming Flickr images are used, we need to define an image size for VQVAE
# Let's set a smaller image size for the VQVAE for computational efficiency

vae_image_size = 64

# Corrected latent_resolution calculation based on ConvEncoder's pooling layers
# ConvEncoder applies 3 MaxPool layers (divide by 2**3 = 8)
latent_resolution = vae_image_size // 8 # Corrected to 8
latent_shape = (latent_channels, latent_resolution, latent_resolution)

plot_alpha = 0.0
is_dark = True
if is_dark:
    pyplot_context = 'dark_background'
    binary_cmap = LinearSegmentedColormap.from_list(name='binary_alpha', colors=[(0,0,0,plot_alpha), (1,1,1,1)])
else:
    pyplot_context = 'default'
    binary_cmap = LinearSegmentedColormap.from_list(name='binary_alpha', colors=[(1,1,1,1), (0,0,0,plot_alpha)])

In [ ]:
class FlickrVAEImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_filenames, image_path, transforms):
        self.image_filenames = image_filenames
        self.image_path = image_path
        self.transforms = transforms

    def __getitem__(self, idx):
        image = cv2.imread(f"{self.image_path}/{self.image_filenames[idx]}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transforms(image=image)['image']
        return torch.from_numpy(image).permute(2, 0, 1).float() # HWC to CHW

    def __len__(self):
        return len(self.image_filenames)


def get_vae_transforms(image_size):
    return A.Compose(
        [
            A.Resize(image_size, image_size),
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), max_pixel_value=255.0), # Normalize to [-1, 1]
        ]
    )

vae_train_dataset = FlickrVAEImageDataset(
    image_filenames=train_df['image'].values,
    image_path=CFG.image_path,
    transforms=get_vae_transforms(vae_image_size)
)
vae_val_dataset = FlickrVAEImageDataset(
    image_filenames=valid_df['image'].values,
    image_path=CFG.image_path,
    transforms=get_vae_transforms(vae_image_size)
)

vae_train_loader = torch.utils.data.DataLoader(
    vae_train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=CFG.num_workers # Re-use num_workers from CFG
)
vae_val_loader = torch.utils.data.DataLoader(
    vae_val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=CFG.num_workers
)

In [ ]:
class ConvBlock(torch.nn.Module):
    def __init__(self, fin, fout, *args, **kwargs):
        super(ConvBlock, self).__init__()
        self._conv = torch.nn.Conv2d(fin, fout, *args, **kwargs)
        self._norm = torch.nn.BatchNorm2d(fout)
        self._relu = torch.nn.LeakyReLU()

    def forward(self, x):
        return self._relu(self._norm(self._conv(x)))

class ConvEncoder(torch.nn.Module):
    def __init__(self, features):
        super(ConvEncoder, self).__init__()

        layers   = []
        for i in range(len(features)-1):
            fi = features[i]
            fo = features[i+1]
            if i > 0:
                layers.append(torch.nn.Sequential(
                    torch.nn.MaxPool2d(2),
                    ConvBlock(fi, fo, 3, padding='same'),
                    ConvBlock(fo, fo, 3, padding='same'),
                ))
            else:
                layers.append(torch.nn.Sequential(
                    ConvBlock(fi, fo, 3, padding='same'),
                    ConvBlock(fo, fo, 3, padding='same'),
                ))
        self.features = features
        self.layers = torch.nn.ModuleList(layers)


    def forward(self, x):
        y = torch.clone(x)
        for layer in self.layers:
            y = layer(y)
        return y

class ConvDecoder(torch.nn.Module):
    def __init__(self, features):
        super(ConvDecoder, self).__init__()

        layers = []
        for i in range(len(features)-1):
            layer = []

            fi = features[i]
            fo = features[i+1]

            if i > 0:
                layer += [
                    torch.nn.Upsample(scale_factor=2),#, mode='bilinear'),
                    ConvBlock(fi, fi, 3, padding='same'),
                ]

            if i < len(features)-2:
                layer += [
                    ConvBlock(fi, fi, 3, padding='same'),
                    ConvBlock(fi, fo, 3, padding='same'),
                ]
            else:
                layer += [
                    ConvBlock(fi, fi, 3, padding='same'),
                    torch.nn.Conv2d(fi, fo, 3, padding='same'),
                ]


            layers.append(torch.nn.Sequential(*layer))

        self.layers = torch.nn.ModuleList(layers)


    def forward(self, x):
        y = torch.clone(x)
        for layer in self.layers:
            y = layer(y)
        return y

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, beta):
        super().__init__()
        self.num_embeddings = num_embeddings # Number of codebook entries
        self.embedding_dim = embedding_dim   # Dimension of each code entry
        self.beta = beta                     # Weight for commitment loss

        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        # Initialize embeddings uniformly
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, z_e): # z_e: continuous latent from encoder (B, C, H, W)
        # Reshape to (B*H*W, C) for distance calculation
        flat_z_e = z_e.permute(0, 2, 3, 1).contiguous().view(-1, self.embedding_dim)

        # Calculate L2 distances between z_e and codebook embeddings
        # (B*H*W, 1) + (num_embeddings,) - 2 * (B*H*W, num_embeddings)
        distances = (torch.sum(flat_z_e**2, dim=1, keepdim=True)
                    + torch.sum(self.embedding.weight**2, dim=1)
                    - 2 * torch.matmul(flat_z_e, self.embedding.weight.t()))

        # Find the closest codebook embedding index for each z_e vector
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)

        # Convert indices to one-hot vectors for explicit selection (optional, can use indexing directly)
        encodings = torch.zeros(encoding_indices.shape[0], self.num_embeddings, device=z_e.device)
        encodings.scatter_(1, encoding_indices, 1)

        # Quantize: z_q is the selected codebook vector (from codebook)
        z_q = torch.matmul(encodings, self.embedding.weight).view(z_e.shape)

        # VQ-VAE Loss components
        # Commitment loss: encoder learns to output vectors close to codebook
        # codebook loss: codebook vectors learn to move towards encoder output (updates codebook)
        # The fix is here: ensure z_q and z_e have the same shape (B, C, H, W) before MSE loss
        vq_loss = F.mse_loss(z_q.detach(), z_e) + self.beta * F.mse_loss(z_q, z_e.detach())

        # Straight-Through Estimator:
        # During backward pass, gradients pass directly through z_q to z_e.
        # This effectively means z_q = z_e in terms of gradients for the encoder.
        z_q = z_e + (z_q - z_e).detach()

        return z_q, vq_loss, encoding_indices # Return quantized latent, VQ loss, and indices

In [ ]:
class VQVAE(nn.Module):
    def __init__(
        self,
        encoder_features,
        decoder_features,
        latent_channels,
        num_embeddings,
        embedding_dim,
        beta
    ):
        super().__init__()
        self.encoder = ConvEncoder(encoder_features + [latent_channels,])
        self.quantizer = VectorQuantizer(num_embeddings, embedding_dim, beta)
        self.decoder = ConvDecoder([latent_channels,] + decoder_features[::-1])

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, _ = self.quantizer(z_e)
        x_recon = self.decoder(z_q)

        reconstruction_loss = F.mse_loss(x_recon, x)
        total_loss = reconstruction_loss + vq_loss

        return total_loss, reconstruction_loss, vq_loss

# VQ-VAE 모델 인스턴스화 (num_embeddings와 embedding_dim은 필요에 따라 조정)
# num_embeddings: 코드북 엔트리의 개수
# embedding_dim: 각 코드 엔트리의 차원 (latent_channels와 동일하게 설정하는 것이 일반적)

num_embeddings = 512 # 예시 값
embedding_dim = latent_channels # latent_channels와 동일

vqvae_model = VQVAE(
    encoder_features=features,
    decoder_features=features,
    latent_channels=latent_channels,
    num_embeddings=num_embeddings,
    embedding_dim=embedding_dim,
    beta=beta_vae
).to(device)

print('Number of parameters (VQ-VAE):', np.sum([torch.numel(x) for x in list(vqvae_model.parameters())]))

### 기존 학습된 VAE 활용
If you have already trained the VQVAE, load it.

In [ ]:
# # If your VAE is saved to the local drive, you can skip loading it from your google drive.
!cp "/content/drive/MyDrive/Colab Notebooks/autoencoder/vqvae_model.pt" .

vqvae_model.load_state_dict(torch.load('vqvae_model.pt', map_location=device))
print('Number of parameters (VQ-VAE):', np.sum([torch.numel(x) for x in list(vqvae_model.parameters())]))

### 추가 학습이 필요한 경우 연속하여 학습하고 저장
Let us train the created VQVAE.

In [ ]:
learning_rate = 5e-4 # Reduced learning rate for VQ-VAE training
optimizer = torch.optim.Adam(vqvae_model.parameters(), learning_rate)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.8, patience=3)

print('Number of parameters (VQ-VAE):', np.sum([torch.numel(x) for x in list(vqvae_model.parameters())]))

In [ ]:
def vae_valid_epoch(model, val_loader, device):
    model.eval()
    total_loss = 0.0
    total_rec_loss = 0.0
    total_vq_loss = 0.0
    with torch.no_grad():
        tqdm_object_val = tqdm(val_loader, desc="VQ-VAE Valid", leave=False)
        for x in tqdm_object_val:
            x = x.to(device)
            loss, reconstruction_loss, vq_loss = model(x)
            total_loss += loss.item()
            total_rec_loss += reconstruction_loss.item()
            total_vq_loss += vq_loss.item()

    return total_loss / len(val_loader), total_rec_loss / len(val_loader), total_vq_loss / len(val_loader)

In [ ]:
num_epochs    = 1
avg_loss_acc = 0.0
mean_rec_acc = 0.0
mean_vq_acc  = 0.0
best_val_loss = float('inf')
for i in range(num_epochs):
    vqvae_model.train()
    tqdm_object = tqdm(vae_train_loader, desc=f"Epoch {i+1} (VQ-VAE Train)", leave=False)
    current_epoch_train_total_loss = 0.0
    current_epoch_train_rec_loss = 0.0
    current_epoch_train_vq_loss = 0.0

    for batch_idx, x in enumerate(tqdm_object):
        x = x.to(device)
        total_loss, reconstruction_loss, vq_loss = vqvae_model(x)

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        current_epoch_train_total_loss += total_loss.item()
        current_epoch_train_rec_loss += reconstruction_loss.item()
        current_epoch_train_vq_loss  += vq_loss.item()

    num_batches = len(vae_train_loader)
    avg_train_total_loss = current_epoch_train_total_loss / num_batches
    avg_train_rec_loss = current_epoch_train_rec_loss / num_batches
    avg_train_vq_loss  = current_epoch_train_vq_loss / num_batches

    # Validation step
    avg_val_total_loss, avg_val_rec_loss, avg_val_vq_loss = vae_valid_epoch(vqvae_model, vae_val_loader, device)

    # Scheduler step
    lr_scheduler.step(avg_val_total_loss)

    print(f"Epoch {i+1}: Train Total Loss = {avg_train_total_loss:.4f}, Train Rec_Loss = {avg_train_rec_loss:.4f}, Train VQ_Loss = {avg_train_vq_loss:.4f}, Val Total Loss = {avg_val_total_loss:.4f}, Val Rec_Loss = {avg_val_rec_loss:.4f}, Val VQ_Loss = {avg_val_vq_loss:.4f}, LR = {get_lr(optimizer):.6f}")

    if avg_val_total_loss < best_val_loss:
        best_val_loss = avg_val_total_loss
        torch.save(vqvae_model.state_dict(), 'vqvae_model.pt') # Save the entire VQVAE model
        print("Saved Best VQ-VAE Model!")

Let's now save the model, and copy the saved model to your google drive.

In [ ]:
torch.save(vqvae_model.state_dict(), 'vqvae_model.pt')

os.makedirs('/content/drive/MyDrive/Colab Notebooks/autoencoder/', exist_ok=True)
!cp vqvae_model.pt "/content/drive/MyDrive/Colab Notebooks/autoencoder/"

Let's make sure that our VQVAE can reconstruct Flickr images. Let's plot a few original and reconstructed Flickr images.

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(10, 5))
perm = torch.randperm(len(vae_val_dataset))[:8]
d = torch.stack([vae_val_dataset[i] for i in perm]).to(device)

with torch.no_grad():
    vqvae_model.eval() # Use the VQ-VAE model in eval mode

    # Get the quantized latent representation
    z_e = vqvae_model.encoder(d)
    z_q, _, _ = vqvae_model.quantizer(z_e) # Quantize the encoder output

    # Reconstruct images using the decoder
    pred = vqvae_model.decoder(z_q)
    lat  = z_q # The latent representation for visualization is z_q

# Denormalize images for plotting from [-1, 1] to [0, 1]
def denormalize_image(img_tensor):
    # Ensure values are clipped to [0, 1] for imshow
    return np.clip(((img_tensor / 2.0) + 0.5).permute(0, 2, 3, 1).cpu().numpy(), 0, 1)

original_images = denormalize_image(d)
reconstructed_images = denormalize_image(pred)
latent_maps = lat.cpu().numpy()

for i in range(8):
    axes[0, i].imshow(original_images[i])
    axes[0, i].axis('off')
    axes[0, i].set_title('Original')

    axes[1, i].imshow(reconstructed_images[i])
    axes[1, i].axis('off')
    axes[1, i].set_title('Reconstructed')

    # For latent representation, visualize one channel or average
    latent_viz = latent_maps[i, 0, :, :] # Use the first channel of the quantized latent for visualization

    # Normalize latent_viz for display
    if latent_viz.max() - latent_viz.min() > 0: # Avoid division by zero if all values are the same
        latent_viz = (latent_viz - np.min(latent_viz)) / (np.max(latent_viz) - np.min(latent_viz))
    else:
        latent_viz = np.zeros_like(latent_viz) # If all values are the same, show as black

    axes[2, i].imshow(latent_viz, cmap='gray')
    axes[2, i].axis('off')
    axes[2, i].set_title('Latent (Ch 0)')

plt.tight_layout()
plt.show()

### VQVAE를 사용하여 잠재 공간의 데이타 생성
Now we need to *generate the latent variables* using the *trained VQVAE*.

In [ ]:
# Prepare latent data for flow matching model (after VQ-VAE training)
vqvae_model.eval()

# Train data latents
lat_train = []
with torch.no_grad():
    for batch in tqdm(vae_train_loader, desc="Generating train latents"):
        x = batch.to(device)
        z_e = vqvae_model.encoder(x) # VQ-VAE 인코더에서 나온 연속적인 잠재 표현
        # Flow Matching 모델은 VQ-VAE의 양자화된 잠재 공간에서 학습하도록 설계되었습니다.
        # 따라서, 인코더 출력(z_e)을 양자화한 z_q를 잠재 변수로 사용합니다.
        z_q, _, _ = vqvae_model.quantizer(z_e) # z_e를 양자화하여 이산적인 잠재 표현(z_q) 얻기
        lat_train.append(z_q.cpu())

lat_train = torch.cat(lat_train)

# Validation data latents
lat_val = []
with torch.no_grad():
    for batch in tqdm(vae_val_loader, desc="Generating validation latents"):
        x = batch.to(device)
        z_e = vqvae_model.encoder(x)
        z_q, _, _ = vqvae_model.quantizer(z_e)
        lat_val.append(z_q.cpu())

lat_val = torch.cat(lat_val)

## Flow Matching Model

Flow Matching is a class of generative models that learns a continuous-time transformation between a simple noise distribution (e.g., standard Gaussian) and a complex data distribution. Unlike Diffusion Models that typically learn a score function or a denoising function to reverse a stochastic process, Flow Matching directly learns a vector field `v(x, t)` that governs a deterministic Ordinary Differential Equation (ODE):

$$ \frac{dx}{dt} = v(x, t) $$

This ODE defines a flow that transports samples from one distribution to another. The training objective is to match the learned vector field to a simpler, analytically tractable target vector field that transports samples along a pre-defined path (e.g., a straight line) between noise and data. This often makes sampling faster and more stable compared to some diffusion processes, as it's a deterministic process rather than a stochastic one during inference.

Now let's build the latent model architecture. We start by defining a positional embedding for the time variable.

In [ ]:
class SinusoidalPositionEmbedding(torch.nn.Module):
    def __init__(self, dim=16, scale=10000.0):
        super().__init__()
        half_dim = dim // 2
        emb_scale = math.log(scale) / (half_dim - 1)
        self.emb_factor = torch.exp(-emb_scale * torch.arange(half_dim, device=device))

    def forward(self, time):
        embeddings = time[:, None] * self.emb_factor[None,:]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

### Transformer 기반 모델 아키텍처

Transformer 기반의 모델을 정의합니다. 다음은 핵심 구성 요소인 `PatchEmbedding`과 `TransformerBlock` 정의입니다.

In [ ]:
class PatchEmbedding(torch.nn.Module):
    def __init__(self, in_channels, patch_size, embed_dim, latent_resolution):
        super().__init__()
        # Patching and linear projection
        self.patch_size = patch_size
        self.proj = torch.nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # Calculate number of patches
        self.num_patches = (latent_resolution // patch_size) ** 2

        # Learnable positional embeddings
        self.position_embedding = torch.nn.Parameter(torch.randn(1, self.num_patches, embed_dim))

    def forward(self, x):
        # x: (B, C, H, W)
        # Convert to patches: (B, E, H/P, W/P)
        x = self.proj(x)
        # Flatten patches: (B, E, N_patches) -> (B, N_patches, E)
        x = x.flatten(2).transpose(1, 2)
        # Add positional embedding
        x = x + self.position_embedding
        return x

class TransformerBlock(torch.nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4., dropout=0.1):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(embed_dim)
        self.attn = torch.nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = torch.nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, mlp_hidden_dim),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(mlp_hidden_dim, embed_dim),
            torch.nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

### 텍스트 조건부 생성을 위한 `LatentFlowTransformerModel` 수정

이 모델은 CLIP의 텍스트 임베딩을 조건으로 받아들이도록 수정되었습니다. `text_conditioning_projection`을 사용하여 CLIP 텍스트 임베딩을 처리합니다.

In [ ]:
class ConditionedLatentFlowTransformerModel(torch.nn.Module):
    def __init__(
        self,
        latent_channels,
        latent_resolution,
        embed_dim,  # Dimension of Transformer tokens
        patch_size,  # Size of image patches (e.g., 2 for 2x2 patches)
        num_transformer_blocks,
        num_heads,
        time_embedding_dim,
        text_conditioning_dim,  # Dimension of the CLIP text embedding
        dropout=0.1,
    ):
        super().__init__()

        self.latent_channels = latent_channels
        self.latent_resolution = latent_resolution
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.num_patches = (latent_resolution // patch_size) ** 2

        # 1. Patch Embedding
        self.patch_embed = PatchEmbedding(latent_channels, patch_size, embed_dim, latent_resolution)

        # 2. Time Embedding
        self.time_embedding = torch.nn.Sequential(
            SinusoidalPositionEmbedding(time_embedding_dim),
            torch.nn.Linear(time_embedding_dim, embed_dim),
            torch.nn.GELU(),
            torch.nn.Linear(embed_dim, embed_dim),
        )

        # 3. Text Conditioning Projection
        self.text_conditioning_projection = torch.nn.Sequential(
            torch.nn.Linear(text_conditioning_dim, embed_dim),
            torch.nn.GELU(),
            torch.nn.Linear(embed_dim, embed_dim),
        )

        # 4. Transformer Blocks
        self.transformer_blocks = torch.nn.ModuleList(
            [
                TransformerBlock(embed_dim, num_heads, dropout=dropout)
                for _ in range(num_transformer_blocks)
            ]
        )

        # 5. Output Reconstruction Head (to predict vector field)
        self.norm_final = torch.nn.LayerNorm(embed_dim)
        self.head = torch.nn.Linear(embed_dim, patch_size * patch_size * latent_channels)

    def forward(self, x, t, text_embedding=None):
        # x: (B, C, H, W) e.g., (B, 2, 4, 4)

        # Patch and Positional Embeddings
        x_tokens = self.patch_embed(x)  # (B, N_patches, embed_dim)

        # Time Embedding (B, embed_dim) -> (B, 1, embed_dim)
        t_emb = self.time_embedding(t).unsqueeze(1)

        # Text Conditioning Embedding (B, embed_dim) -> (B, 1, embed_dim)
        if text_embedding is not None:
            text_cond_emb = self.text_conditioning_projection(text_embedding).unsqueeze(1)
        else:
            # Unconditional generation if no text_embedding is provided
            text_cond_emb = torch.zeros(x.shape[0], 1, self.embed_dim, device=x.device)

        # Add conditional embeddings to all patch tokens
        x_tokens = x_tokens + t_emb + text_cond_emb

        # Pass through Transformer blocks
        for block in self.transformer_blocks:
            x_tokens = block(x_tokens)

        # Normalize and reconstruct output
        x_tokens = self.norm_final(x_tokens)  # (B, N_patches, embed_dim)

        # Project back to patch representation
        v_field_patches = self.head(x_tokens)  # (B, N_patches, patch_size*patch_size*latent_channels)

        # Reshape to spatial grid
        v_field_patches = v_field_patches.reshape(
            x.shape[0],
            self.num_patches,
            self.latent_channels,
            self.patch_size,
            self.patch_size,
        )

        # Arrange patches back into an image
        h_grid = w_grid = int(self.num_patches**0.5)
        v_field = torch.zeros(
            x.shape[0],
            self.latent_channels,
            self.latent_resolution,
            self.latent_resolution,
            device=x.device,
        )

        patch_idx = 0
        for i in range(h_grid):
            for j in range(w_grid):
                v_field[
                    :, :, i * self.patch_size : (i + 1) * self.patch_size, j * self.patch_size : (j + 1) * self.patch_size
                ] = v_field_patches[:, patch_idx, :, :, :]
                patch_idx += 1

        return v_field


Transformer 관련 변수 설정 (Flickr Latent Space)

In [ ]:
# New hyperparameters for conditional model (using text conditioning)
# Transformer specific hyperparameters

time_embedding_dim = 128
learning_rate = 1e-3
embed_dim = 128 # Dimension of Transformer tokens
patch_size = 2 # Size of image patches
num_transformer_blocks = 4 # Number of Transformer blocks
num_heads = 4 # Number of attention heads

flow_matching_model = ConditionedLatentFlowTransformerModel(
     latent_channels=latent_channels,
     latent_resolution=latent_resolution, # Use the new latent_resolution from Flickr VAE
     embed_dim=embed_dim,
     patch_size=patch_size,
     num_transformer_blocks=num_transformer_blocks,
     num_heads=num_heads,
     time_embedding_dim=time_embedding_dim,
     text_conditioning_dim=CFG.projection_dim # Pass the text conditioning dimension
 )
flow_matching_model = flow_matching_model.to(device)
print('Number of parameters (Transformer):', np.sum([torch.numel(x) for x in list(flow_matching_model.parameters())]))
print(f"flow_matching_model.patch_embed.num_patches: {flow_matching_model.patch_embed.num_patches}")

학습된 transfomer 모델 loading

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/flow/flow_matching_model_transformer.pt" .
flow_matching_model.load_state_dict(torch.load('flow_matching_model_transformer.pt', map_location=device))
flow_matching_model = flow_matching_model.to(device)
print('Number of parameters (Transformer):', np.sum([torch.numel(x) for x in list(flow_matching_model.parameters())]))

### 필요하면 추가 학습


In [ ]:
def do_step(model, x_0, y=None, weighting_function=None): # Added weighting_function parameter
    # Sample t uniformly from [0, 1)
    t_float = torch.rand(x_0.shape[0], device=device).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)

    # Sample x_1 from a standard Gaussian (noise)
    x_1 = torch.normal(0, 1, size=x_0.shape, device=device)

    # Define the straight path: x_t = (1-t)x_0 + t*x_1
    x_t = (1.0 - t_float) * x_0 + t_float * x_1

    # Target vector field for a straight path is x_1 - x_0
    v_target = x_1 - x_0

    # Model predicts the vector field v_theta. Pass t_float as a 1D tensor.
    # For Flickr, we likely don't have class labels unless explicitly added.
    # So, we pass y=None, and the model handles unconditional generation.
    v_theta = model(x_t, t_float.squeeze(-1).squeeze(-1).squeeze(-1), text_embedding=y)

    # Calculate the squared error (element-wise)
    squared_error = (v_theta - v_target)**2

    # Apply weighting if a weighting function is provided
    if weighting_function is not None:
        # Ensure the weight tensor has the correct shape for broadcasting
        weights = weighting_function(t_float)
        loss = torch.mean(weights * squared_error) # Apply weights and then take mean
    else:
        # Original Flow Matching loss: minimize the squared difference (mean over batch and dimensions)
        loss = torch.mean(squared_error)

    return loss

With this, we can start training it.

In [ ]:
num_epochs    = 1
i_log = 1 # Changed i_log to 1 to ensure losses are recorded

optimizer = torch.optim.Adam(flow_matching_model.parameters(), learning_rate)

# num_batches will be based on lat_train, which is generated from Flickr images
num_batches = int(math.ceil(lat_train.shape[0] / batch_size))
num_batches_val = int(math.ceil(lat_val.shape[0] / batch_size))

# Example weighting function (e.g., giving more weight to later time steps)
def example_weighting_function(t):
    return 1.0 + 2.0 * t  # Example: linearly increasing weight with time

losses = []
for i in range(num_epochs):
    flow_matching_model.train()
    train_ids = torch.randperm(lat_train.shape[0])
    average_loss = 0.0
    for bid in range(num_batches):

        # Get batch of latents from lat_train
        batch_ids = train_ids[bid*batch_size:(bid+1)*batch_size]
        x = lat_train[batch_ids,...].to(device)
        # No y (class label) for unconditional generation
        y = None

        # Pass the weighting function to do_step
        loss = do_step(flow_matching_model, x, y, weighting_function=example_weighting_function)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        average_loss += loss.cpu().detach().numpy() / num_batches

    if (i + 1) % i_log == 0:
        flow_matching_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bid in range(num_batches_val):
                x = lat_val[bid*batch_size:(bid+1)*batch_size,...].to(device)
                y = None # No y for unconditional generation
                # Pass the weighting function for validation loss as well
                loss = do_step(flow_matching_model, x, y, weighting_function=example_weighting_function)
                val_loss += loss.cpu().detach().numpy()

        val_loss /= num_batches_val
        losses.append([average_loss, val_loss])
        print(f'Epoch {i} loss = {average_loss}, val_loss = {val_loss}')

### Training and Validation Loss Graph

In [ ]:
losses = np.array(losses)
x = np.arange(losses.shape[0]) * i_log
with plt.style.context(pyplot_context):
    fig, axes = plt.subplots(figsize=(7,5))
    axes.plot(x, losses[:,0], label='train')
    axes.plot(x, losses[:,1], label='validation')
    axes.patch.set_alpha(plot_alpha)
    fig.patch.set_alpha(plot_alpha)
    axes.legend(frameon=False)

Save the model and copy to google drive

In [ ]:
torch.save(flow_matching_model.state_dict(), 'flow_matching_model_transformer.pt') # Save with a new name
os.makedirs('/content/drive/MyDrive/Colab Notebooks/flow/', exist_ok=True)
!cp flow_matching_model_transformer.pt "/content/drive/MyDrive/Colab Notebooks/flow/"

## Flow Model Evaluation
### Model Sampling

In [ ]:
def do_flow_matching_sampling(model, x_1, num_sampling_steps, y=None): # y can be None for unconditional
    x_t = torch.clone(x_1) # Start from noise (x_1 at t=1)
    dt = 1.0 / num_sampling_steps

    model.eval()
    with torch.no_grad():
        for i in range(num_sampling_steps):
            t_float_current = 1.0 - i * dt  # Current time t for x_t

            # Ensure time tensors match the batch size and expected shape for the model (batch_size,)
            # t_float_current is a scalar. torch.ones(x_t.shape[0], device=device) creates a 1D tensor of batch_size.
            # Multiplying them correctly creates a 1D tensor of shape (batch_size,).
            t_current_tensor = t_float_current * torch.ones(x_t.shape[0], device=device)

            # RK4 steps for backward integration
            # k1 = dt * v(x_t, t)
            v1 = model(x_t, t_current_tensor, y) # Pass the 1D tensor directly
            k1 = v1 * dt

            # k2 = dt * v(x_t - 0.5 * k1, t - 0.5 * dt)
            x_k2_input = x_t - 0.5 * k1
            t_k2_float = t_float_current - 0.5 * dt
            t_k2_tensor = t_k2_float * torch.ones(x_t.shape[0], device=device) # Corrected
            v2 = model(x_k2_input, t_k2_tensor, y) # Pass the 1D tensor directly
            k2 = v2 * dt

            # k3 = dt * v(x_t - 0.5 * k2, t - 0.5 * dt)
            x_k3_input = x_t - 0.5 * k2
            t_k3_float = t_float_current - 0.5 * dt # Same time as t_k2_float
            t_k3_tensor = t_k3_float * torch.ones(x_t.shape[0], device=device) # Corrected
            v3 = model(x_k3_input, t_k3_tensor, y) # Pass the 1D tensor directly
            k3 = v3 * dt

            # k4 = dt * v(x_t - k3, t - dt)
            x_k4_input = x_t - k3
            t_k4_float = t_float_current - dt
            t_k4_tensor = t_k4_float * torch.ones(x_t.shape[0], device=device) # Corrected
            v4 = model(x_k4_input, t_k4_tensor, y) # Pass the 1D tensor directly
            k4 = v4 * dt

            # Update x_t using the RK4 formula for backward integration
            x_t = x_t - (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
    return x_t

### 텍스트 조건을 이용한 이미지 생성 함수

이 함수는 CLIP 모델을 사용하여 텍스트 설명을 임베딩으로 변환하고, 이 임베딩을 수정된 Flow Matching 모델에 전달하여 이미지 잠재 변수를 생성한 다음, VQVAE 디코더를 사용하여 실제 이미지로 디코딩합니다.


In [ ]:
def generate_images_from_text(text_query, clip_model_instance, flow_model_instance, num_images=4, num_sampling_steps=50):
    # 1. CLIP 모델을 사용하여 텍스트 임베딩 생성
    encoded_query = CFG.tokenizer([text_query], return_tensors='pt', padding=True, truncation=True, max_length=CFG.max_length)

    with torch.no_grad():
        clip_text_features = clip_model_instance.text_encoder(
            input_ids=encoded_query['input_ids'].to(CFG.device),
            attention_mask=encoded_query['attention_mask'].to(CFG.device)
        )
        clip_text_embedding = clip_model_instance.text_projection(clip_text_features) # (1, CFG.projection_dim)

    # 텍스트 임베딩을 num_images 배치 크기에 맞게 복제
    text_embedding_batch = clip_text_embedding.repeat(num_images, 1)

    # 2. Flow Matching 모델을 사용하여 잠재 변수 생성
    initial_noise = torch.normal(0, 1, size=(num_images,) + latent_shape, device=device)

    # Use the pre-defined do_flow_matching_sampling function (RK4 integrator)
    generated_latents = do_flow_matching_sampling(
        flow_model_instance, initial_noise, num_sampling_steps, y=text_embedding_batch
    )

    # 3. VAE 디코더를 사용하여 이미지 디코딩
    # Corrected: Access decoder through vqvae_model
    vqvae_model.decoder.eval()
    with torch.no_grad():
        # Denormalize and clip values to [0, 1] for imshow
        generated_images = np.clip(((vqvae_model.decoder(generated_latents) / 2.0) + 0.5).permute(0, 2, 3, 1).cpu().numpy(), 0, 1)

    # 4. 결과 시각화
    fig, axes = plt.subplots(1, num_images, figsize=(num_images * 3, 3))
    if num_images == 1:
        axes = [axes]
    for i, img in enumerate(generated_images):
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f"Generated {i+1}")
    plt.suptitle(f"Query: '{text_query}'", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
text_query_input = input("생성하고 싶은 이미지에 대한 텍스트 설명을 입력하세요: ")
generate_images_from_text(text_query_input, clip_model, flow_matching_model, num_images=4)